# Embedding shift experiment 
In this file we compare our results with the embedding shift experiments of the seminal modality gap paper: "Mind the Gap: Understanding the Modality Gap in Multi-modal Contrastive Representation Learning"

The embedding shift works like this:
- gap defined as the difference of the mean embedding between each couple modality
- x_new = x - λ*gap and y_new = y - λ*gap, we set λ=1 to totally remove the gap

After shifting we compute new gap measures, retrieval, clustering and classification

## MSCOCO with ImageNet Labels — Embedding Shift (Translation)

This section reproduces the embedding shift experiment on the **filtered MSCOCO split with ImageNet labels**.

Filtering follows the same rule used in `method_verification.ipynb`:
- keep only classes that appear in the validation set
- and have at least **10 training samples**

Then, following the same pipeline used for the other datasets in this notebook, we:
- collect filtered test embeddings
- compute the translation gap
- apply the shift with $\lambda = 1$
- evaluate retrieval, gap metrics, and clustering before and after the shift


In [1]:
import torch
from torch.utils.data import DataLoader, Subset
import os, sys, random
import numpy as np
import torch.nn.functional as F
import pandas as pd
sys.path.append(os.path.abspath(".."))
from tqdm.auto import tqdm
from metrics.retrieval import compute_retrieval
from analysis.modality_gap import compute_gap
from metrics.clustering import clustering_metrics_two_modalities_mscoco_imagenet_labels
from dataset.mscoco.mscoco_dataloader_with_imagenet_labels import (
    MSCOCOEmbeddingsDatasetWithImageNetLabels,
    mscoco_imagenet_collate_fn,
)

seed = 123
SEED = seed
CLIP_MODEL = "ViT-L-16-SigLIP2-256"
CLIP_PRETRAINED = "webli"
MODEL_NAME = f"{CLIP_MODEL}___{CLIP_PRETRAINED}"

g = torch.Generator().manual_seed(seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

def _resolve_flickr_precomputed_dir(model_name: str) -> str:
    root = "/mnt/media/emanuele/few_dimensions/dataset/flickr30k/precomputed_embeddings_with_labels"
    candidates = [
        os.path.join(root, model_name),
        os.path.join(
            root,
            model_name.lower().replace("vit-b-32", "clip_vit_b_32").replace("vit-b-16", "clip_vit_b_16"),
        ),
    ]
    for candidate in candidates:
        if os.path.isdir(candidate):
            return candidate
    return candidates[0]


if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

DEVICE = "cuda:1" if torch.cuda.is_available() else "cpu"

MSCOCO_MODEL_ROOT = f"/mnt/media/emanuele/few_dimensions/dataset/mscoco/data/mscoco/{MODEL_NAME}"
MSCOCO_IMAGENET_TRAIN_DIR = f"{MSCOCO_MODEL_ROOT}/precomputed_train2017_clip_imagenet"
MSCOCO_IMAGENET_TEST_DIR = f"{MSCOCO_MODEL_ROOT}/precomputed_val2017_clip_imagenet"
MSCOCO_IMAGENET_DATASET_NAME = "mscoco_imagenet_labels"
MSCOCO_IMAGENET_MIN_TRAIN_SAMPLES = 10

PRECOMPUTED_TRAIN_DIR = f"/mnt/media/emanuele/few_dimensions/dataset/msrvtt/{MODEL_NAME}_v2/precomputed_train"
PRECOMPUTED_TEST_DIR = f"/mnt/media/emanuele/few_dimensions/dataset/msrvtt/{MODEL_NAME}_v2/precomputed_test"
FLICKR_PRECOMPUTED_DIR = _resolve_flickr_precomputed_dir(MODEL_NAME)

mscoco_train_ds = MSCOCOEmbeddingsDatasetWithImageNetLabels(
    MSCOCO_IMAGENET_TRAIN_DIR,
    split_name="train_shard",
    return_label_name=False,
)

mscoco_test_ds = MSCOCOEmbeddingsDatasetWithImageNetLabels(
    MSCOCO_IMAGENET_TEST_DIR,
    split_name="val_shard",
    return_label_name=False,
)


def _count_labels(dataset):
    counts = {}
    for idx in tqdm(range(len(dataset)), desc="Counting labels"):
        _, _, label_value = dataset[idx]
        label_value = int(label_value.item()) if torch.is_tensor(label_value) else int(label_value)
        counts[label_value] = counts.get(label_value, 0) + 1
    return counts


mscoco_train_counts = _count_labels(mscoco_train_ds)
mscoco_test_counts = _count_labels(mscoco_test_ds)

mscoco_test_classes = set(mscoco_test_counts.keys())
mscoco_keep_classes = {
    class_id
    for class_id in mscoco_test_classes
    if mscoco_train_counts.get(class_id, 0) >= MSCOCO_IMAGENET_MIN_TRAIN_SAMPLES
}

mscoco_train_indices = [
    idx
    for idx in tqdm(range(len(mscoco_train_ds)), desc="Filtering MSCOCO train")
    if int(mscoco_train_ds[idx][2].item()) in mscoco_keep_classes
]

mscoco_test_indices = [
    idx
    for idx in tqdm(range(len(mscoco_test_ds)), desc="Filtering MSCOCO val")
    if int(mscoco_test_ds[idx][2].item()) in mscoco_keep_classes
]

mscoco_filtered_train = Subset(mscoco_train_ds, mscoco_train_indices)
mscoco_filtered_test = Subset(mscoco_test_ds, mscoco_test_indices)


def _count_labels_subset(subset):
    counts = {}
    for idx in tqdm(range(len(subset)), desc="Counting filtered labels"):
        _, _, label_value = subset[idx]
        label_value = int(label_value.item()) if torch.is_tensor(label_value) else int(label_value)
        counts[label_value] = counts.get(label_value, 0) + 1
    return counts


mscoco_filtered_train_counts = _count_labels_subset(mscoco_filtered_train)
mscoco_filtered_test_counts = _count_labels_subset(mscoco_filtered_test)

mscoco_train_classes = set(mscoco_filtered_train_counts.keys())
mscoco_test_classes = set(mscoco_filtered_test_counts.keys())

assert mscoco_train_classes == mscoco_test_classes, "Train/test class mismatch after MSCOCO filtering"

MSCOCO_IMAGENET_N_CLUSTERS = len(mscoco_train_classes)
BATCH_SIZE = 256
NUM_WORKERS = 0

mscoco_loader = DataLoader(
    mscoco_filtered_test,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=mscoco_imagenet_collate_fn,
)

print(f"Using device                   : {DEVICE}")
print(f"Model name                     : {MODEL_NAME}")
print(f"MSCOCO train samples original   : {len(mscoco_train_ds)}")
print(f"MSCOCO val samples original     : {len(mscoco_test_ds)}")
print(f"Filtering threshold             : {MSCOCO_IMAGENET_MIN_TRAIN_SAMPLES}")
print(f"Classes kept                    : {len(mscoco_keep_classes)}")
print(f"Train samples after filtering   : {len(mscoco_filtered_train)}")
print(f"Val samples after filtering     : {len(mscoco_filtered_test)}")
print(f"Clusters inferred from labels   : {MSCOCO_IMAGENET_N_CLUSTERS}")
print(f"Embedding dim                   : {int(mscoco_train_ds[0][0].shape[-1])}")


/opt/anaconda3/envs/few_dim_modalitygap/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Loaded COCO ImageNet] 118287 samples from /mnt/media/emanuele/few_dimensions/dataset/mscoco/data/mscoco/ViT-L-16-SigLIP2-256___webli/precomputed_train2017_clip_imagenet | vision_emb shape=(118287, 1024)
[Loaded COCO ImageNet] 5000 samples from /mnt/media/emanuele/few_dimensions/dataset/mscoco/data/mscoco/ViT-L-16-SigLIP2-256___webli/precomputed_val2017_clip_imagenet | vision_emb shape=(5000, 1024)


Counting filtered labels: 100%|██████████| 4963/4963 [00:00<00:00, 85841.96it/s]

Using device                   : cuda:1
Model name                     : ViT-L-16-SigLIP2-256___webli
MSCOCO train samples original   : 118287
MSCOCO val samples original     : 5000
Filtering threshold             : 10
Classes kept                    : 517
Train samples after filtering   : 113777
Val samples after filtering     : 4963
Clusters inferred from labels   : 517
Embedding dim                   : 1024


In [2]:

def _compute_mean_shift(text_embeddings, vision_embeddings):
    return vision_embeddings.mean(dim=0) - text_embeddings.mean(dim=0)


def _gap_scalar(metric_name, x_text, x_vision):
    value = compute_gap(metric_name, x_text, x_vision, None)
    if isinstance(value, dict):
        if "text_vision" in value:
            value = value["text_vision"]
        else:
            value = next(iter(value.values()))
    if torch.is_tensor(value):
        value = value.item()
    return float(value)


def _normalize_rows(points, eps=1e-12):
    return points / (points.norm(p=2, dim=1, keepdim=True) + eps)


mscoco_text_batches, mscoco_vision_batches, mscoco_label_batches = [], [], []

with torch.no_grad():
    for text_batch, vision_batch, label_batch in tqdm(
        mscoco_loader,
        desc="Collecting filtered MSCOCO-ImageNet val embeddings",
    ):
        mscoco_text_batches.append(F.normalize(text_batch.to(DEVICE), dim=-1).cpu())
        mscoco_vision_batches.append(F.normalize(vision_batch.to(DEVICE), dim=-1).cpu())
        mscoco_label_batches.append(label_batch.detach().cpu())

mscoco_text = torch.cat(mscoco_text_batches, dim=0)
mscoco_vision = torch.cat(mscoco_vision_batches, dim=0)
mscoco_labels = torch.cat(mscoco_label_batches, dim=0)

mscoco_batch_rmg_original = [
    _gap_scalar("RMG", text_batch, vision_batch)
    for text_batch, vision_batch in zip(mscoco_text_batches, mscoco_vision_batches)
]
mscoco_batch_rmg_original_mean = float(np.mean(mscoco_batch_rmg_original))

mscoco_gap = _compute_mean_shift(mscoco_text, mscoco_vision)
mscoco_gap_norm_original = float(torch.norm(mscoco_gap).item())

mscoco_text_shifted = _normalize_rows(mscoco_text + (mscoco_gap / 2))
mscoco_vision_shifted = _normalize_rows(mscoco_vision - (mscoco_gap / 2))
mscoco_batch_rmg_shifted = []
for text_batch, vision_batch in zip(mscoco_text_batches, mscoco_vision_batches):
    batch_gap = _compute_mean_shift(text_batch, vision_batch)
    text_shifted_batch = _normalize_rows(text_batch + (batch_gap / 2))
    vision_shifted_batch = _normalize_rows(vision_batch - (batch_gap / 2))
    mscoco_batch_rmg_shifted.append(_gap_scalar("RMG", text_shifted_batch, vision_shifted_batch))
mscoco_batch_rmg_shifted_mean = float(np.mean(mscoco_batch_rmg_shifted))
mscoco_gap_shifted = _compute_mean_shift(mscoco_text_shifted, mscoco_vision_shifted)
mscoco_gap_norm_shifted = float(torch.norm(mscoco_gap_shifted).item())

mscoco_cluster_original = clustering_metrics_two_modalities_mscoco_imagenet_labels(
    mscoco_text,
    mscoco_vision,
    mscoco_labels,
    n_clusters=MSCOCO_IMAGENET_N_CLUSTERS,
    random_state=SEED,
)
mscoco_cluster_shifted = clustering_metrics_two_modalities_mscoco_imagenet_labels(
    mscoco_text_shifted,
    mscoco_vision_shifted,
    mscoco_labels,
    n_clusters=MSCOCO_IMAGENET_N_CLUSTERS,
    random_state=SEED,
)

mscoco_summary = pd.DataFrame(
    [
        {
            "state": "original",
            "gap_norm": mscoco_gap_norm_original,
            "retrieval@1": float(compute_retrieval(MSCOCO_IMAGENET_DATASET_NAME, (mscoco_text, mscoco_vision), top_k=1)),
            "retrieval@5": float(compute_retrieval(MSCOCO_IMAGENET_DATASET_NAME, (mscoco_text, mscoco_vision), top_k=5)),
            "retrieval@10": float(compute_retrieval(MSCOCO_IMAGENET_DATASET_NAME, (mscoco_text, mscoco_vision), top_k=10)),
            "RMG": mscoco_batch_rmg_original_mean,
            "L2M": _gap_scalar("L2M", mscoco_text, mscoco_vision),
            "L2I": _gap_scalar("L2I", mscoco_text, mscoco_vision),
            "cosineTP": _gap_scalar("cosineTP", mscoco_text, mscoco_vision),
            "V-measure": float(mscoco_cluster_original["V-measure"]),
        },
        {
            "state": "shifted_lam1",
            "gap_norm": mscoco_gap_norm_shifted,
            "retrieval@1": float(compute_retrieval(MSCOCO_IMAGENET_DATASET_NAME, (mscoco_text_shifted, mscoco_vision_shifted), top_k=1)),
            "retrieval@5": float(compute_retrieval(MSCOCO_IMAGENET_DATASET_NAME, (mscoco_text_shifted, mscoco_vision_shifted), top_k=5)),
            "retrieval@10": float(compute_retrieval(MSCOCO_IMAGENET_DATASET_NAME, (mscoco_text_shifted, mscoco_vision_shifted), top_k=10)),
            "RMG": mscoco_batch_rmg_shifted_mean,
            "L2M": _gap_scalar("L2M", mscoco_text_shifted, mscoco_vision_shifted),
            "L2I": _gap_scalar("L2I", mscoco_text_shifted, mscoco_vision_shifted),
            "cosineTP": _gap_scalar("cosineTP", mscoco_text_shifted, mscoco_vision_shifted),
            "V-measure": float(mscoco_cluster_shifted["V-measure"]),
        },
    ]
)

display(mscoco_summary.round(4))
print(f"Filtered val samples used       : {len(mscoco_filtered_test)}")
print(f"Clusters inferred from val set  : {MSCOCO_IMAGENET_N_CLUSTERS}")


,state,gap_norm,retrieval@1,retrieval@5,retrieval@10,RMG,L2M,L2I,cosineTP,V-measure
0,original,0.9948,0.5102,0.7421,0.8213,0.8594,0.9948,1.3030,0.1507,0.6381
1,shifted_lam1,0.0574,0.4229,0.6595,0.7447,0.7166,0.0574,0.9617,0.5348,0.6399


Filtered val samples used       : 4963
Clusters inferred from val set  : 517


## MSRVTT v2 — Embedding Shift (Translation)

### Imports and constants

In [ ]:
import copy
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    homogeneity_score,
    v_measure_score,
)

try:
    import wandb
except ImportError as exc:
    raise ImportError("wandb is required. Install with `pip install wandb`.") from exc

sys.path.append(os.path.abspath(".."))

from dataset.msrvtt.msrvtt_dataloaderv2 import (
    MSRVTTEmbeddingsDatasetV2,
    msrvtt_v2_collate_fn,
)
from analysis.modality_gap import compute_gap
from metrics.retrieval import compute_retrieval
from metrics.clustering import clustering_metrics_two_modalities_msrvtt
from models.fusion_mlp_classifier import LinearProbingIndependentModalities

print("Using device:", DEVICE)
print("Using model :", MODEL_NAME)
print("MSRVTT train:", PRECOMPUTED_TRAIN_DIR)
print("MSRVTT test :", PRECOMPUTED_TEST_DIR)

DIRECTION = "text_to_vision"
N_CLUSTERS = 20
MAX_CLUSTER_SAMPLES = 5000
LAMBDA_SHIFT = 1.0

CLASSIFIER_EPOCHS = 200
CLASSIFIER_PATIENCE = 20
MAX_LR = 3e-4
WEIGHT_DECAY = 1e-4
WANDB_ENABLED = True
WANDB_PROJECT = "msrvtt_v2_translation_verification"
WANDB_GROUP = "embedding_shift"

GAP_NAMES = ["RMG", "L2M", "L2I", "cosineTP"]

BASE_SAVE_DIR = Path("msrvtt_v2_classification_runs") / "translation"
BASE_SAVE_DIR.mkdir(parents=True, exist_ok=True)


### Load MSRVTT v2 datasets and build data loaders

In [ ]:
ds_train = MSRVTTEmbeddingsDatasetV2(
    PRECOMPUTED_TRAIN_DIR,
    split_name="train_shard",
    return_metadata=False,
)

ds_test = MSRVTTEmbeddingsDatasetV2(
    PRECOMPUTED_TEST_DIR,
    split_name="test_shard",
    return_metadata=False,
)

train_loader = DataLoader(
    ds_train,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=msrvtt_v2_collate_fn,
    generator=g,
)

test_loader = DataLoader(
    ds_test,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=msrvtt_v2_collate_fn,
)

EMBEDDING_DIM = int(ds_train[0][0].shape[-1])

print(f"Train samples : {len(ds_train)}")
print(f"Test samples  : {len(ds_test)}")
print(f"Embedding dim : {EMBEDDING_DIM}")

# Build label look-up table for the classifier
def _extract_labels(dataset, desc="labels"):
    labels = []
    for idx in tqdm(range(len(dataset)), desc=desc):
        item = dataset[idx]
        lbl = item[2]
        labels.append(int(lbl.item()) if torch.is_tensor(lbl) else int(lbl))
    return np.asarray(labels, dtype=np.int64)

train_labels = _extract_labels(ds_train, desc="Extracting train labels")
NUM_CLASSES = len(set(train_labels.tolist()))
LABEL_LUT = torch.zeros(int(train_labels.max()) + 1, dtype=torch.long) - 1
for new_idx, orig_label in enumerate(sorted(set(train_labels.tolist()))):
    LABEL_LUT[orig_label] = new_idx

print(f"Number of classes: {NUM_CLASSES}")
print(f"Label LUT size   : {LABEL_LUT.numel()}")


In [ ]:
def compute_l2m_vectors(text_embeddings, vision_embeddings):
    # Compute the L2M vectors for each sample
    mean_text = text_embeddings.mean(dim=0)
    mean_vision = vision_embeddings.mean(dim=0)
    l2m_vector = mean_vision - mean_text
    return l2m_vector

In [ ]:
# ENTIRE test SET ANALYSIS
def normalize_unit_sphere(points, eps=1e-12):
    if isinstance(points, torch.Tensor):
        return points / (points.norm(p=2, dim=1, keepdim=True) + eps)
    return points / (np.linalg.norm(points, ord=2, axis=1, keepdims=True) + eps)


def _gap_scalar(metric_name, x_text, x_vision):
    value = compute_gap(metric_name, x_text, x_vision, None)
    if isinstance(value, dict):
        value = value.get('text_vision', next(iter(value.values())))
    if torch.is_tensor(value):
        value = value.item()
    return float(value)


iter_test = iter(test_loader)
Xt, Xv, y = [], [], []
for batch in iter_test:
    Xt.append(batch[0])
    Xv.append(batch[1])
    y.append(batch[2])

Xt = torch.cat(Xt, dim=0)
Xv = torch.cat(Xv, dim=0)
y = torch.cat(y, dim=0)

gap = compute_l2m_vectors(Xt, Xv)
gap_norm = torch.norm(gap).item()
rmg_original = _gap_scalar('RMG', Xt, Xv)
retrieval_1_original = compute_retrieval('msrvtt',(Xt, Xv, y),top_k=1)
clustering_original = clustering_metrics_two_modalities_msrvtt(Xt, Xv, y, n_clusters=N_CLUSTERS)['V-measure']

x_new = normalize_unit_sphere(Xt + (gap/2))
y_new = normalize_unit_sphere(Xv - (gap/2))
new_gap = torch.norm(compute_l2m_vectors(x_new, y_new)).item()
rmg_new = _gap_scalar('RMG', x_new, y_new)
retrieval_1_new = compute_retrieval('msrvtt',(x_new, y_new, y),top_k=1)
retrieval_5_new = compute_retrieval('msrvtt',(x_new, y_new, y),top_k=5)
retrieval_10_new = compute_retrieval('msrvtt',(x_new, y_new, y),top_k=10)
cosine_tp_new = _gap_scalar('cosineTP', x_new, y_new)
l2i_new = _gap_scalar('L2I', x_new, y_new)

batch_rmg_original = []
batch_rmg_new = []
for batch in test_loader:
    Xt_b, Xv_b, _ = batch[:3]
    batch_rmg_original.append(_gap_scalar('RMG', Xt_b, Xv_b))

    gap_b = compute_l2m_vectors(Xt_b, Xv_b)
    x_new_b = normalize_unit_sphere(Xt_b + (gap_b / 2))
    y_new_b = normalize_unit_sphere(Xv_b - (gap_b / 2))
    batch_rmg_new.append(_gap_scalar('RMG', x_new_b, y_new_b))

mean_batch_rmg_original = float(np.mean(batch_rmg_original))
mean_batch_rmg_new = float(np.mean(batch_rmg_new))

clustering_new = clustering_metrics_two_modalities_msrvtt(x_new, y_new, y, n_clusters=N_CLUSTERS)['V-measure']
print(f"Original gap: {gap_norm:.4f} \t| Clustering v original: {clustering_original.item():.4f}")
print(f"New gap     : {new_gap:.4f} \t| Clustering v new: {clustering_new.item():.4f}")
print(f"RMG [original]: {rmg_original:.4f}")
print(f"RMG [new]: {rmg_new:.4f}")
print(f"Mean batch RMG [original]: {mean_batch_rmg_original:.4f}")
print(f"Mean batch RMG [new]: {mean_batch_rmg_new:.4f}")

print(f"Retrieval@1 [original]: {retrieval_1_original:.4f}")
print(f"Retrieval@1 [new]: {retrieval_1_new:.4f}")
print(f"Retrieval@5 [new]: {retrieval_5_new:.4f}")
print(f"Retrieval@10 [new]: {retrieval_10_new:.4f}")
print(f"Cosine TP [new]: {cosine_tp_new:.4f}")
print(f"L2I [new]: {l2i_new:.4f}")

<!-- Original gap: 0.7806 	| Clustering v original: 0.2448
New gap     : 0.0074 	| Clustering v new: 0.2833
Retrieval@1 [original]: 0.3260
Retrieval@1 [new]: 0.2490
Retrieval@5 [new]: 0.4570
Retrieval@10 [new]: 0.5480
Cosine TP [new]: 0.5130
L2I [new]: 0.9837
/tmp/ipykernel_3427627/4277140992.py:2: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  return points / np.linalg.norm(points,ord=2, axis=1)[:, np.newaxis] -->

In [ ]:
def normalize_unit_sphere(points, eps=1e-12):
    if isinstance(points, torch.Tensor):
        return points / (points.norm(p=2, dim=1, keepdim=True) + eps)
    return points / (np.linalg.norm(points, ord=2, axis=1, keepdims=True) + eps)


def _to_scalar(x):
    if torch.is_tensor(x):
        return float(x.item())
    return float(x)


def _batch_clustering_score(x_text, x_vision, labels):
    n_unique = int(torch.unique(labels).numel())
    n_clusters_batch = min(N_CLUSTERS, n_unique)
    if n_clusters_batch < 2:
        return np.nan

    score = clustering_metrics_two_modalities_msrvtt(
        x_text,
        x_vision,
        labels,
        n_clusters=n_clusters_batch,
    )["V-measure"]
    return _to_scalar(score)


retrieval_before, retrieval_after = [], []
clustering_before, clustering_after = [], []
gap_before, gap_after = [], []

for batch_idx, batch in enumerate(test_loader):
    Xt_b, Xv_b, y_b = batch[:3]

    gap_b = compute_l2m_vectors(Xt_b, Xv_b)
    gap_before.append(float(torch.norm(gap_b).item()))

    r_before = compute_retrieval("msrvtt", (Xt_b, Xv_b, y_b), top_k=1)
    c_before = _batch_clustering_score(Xt_b, Xv_b, y_b)

    x_new_b = normalize_unit_sphere(Xt_b + (gap_b / 2))
    y_new_b = normalize_unit_sphere(Xv_b - (gap_b / 2))

    new_gap_b = compute_l2m_vectors(x_new_b, y_new_b)
    gap_after.append(float(torch.norm(new_gap_b).item()))

    r_after = compute_retrieval("msrvtt", (x_new_b, y_new_b, y_b), top_k=1)
    c_after = _batch_clustering_score(x_new_b, y_new_b, y_b)

    retrieval_before.append(_to_scalar(r_before))
    retrieval_after.append(_to_scalar(r_after))
    clustering_before.append(c_before)
    clustering_after.append(c_after)

avg_gap_before = float(np.mean(gap_before))
avg_gap_after = float(np.mean(gap_after))
avg_retrieval_before = float(np.mean(retrieval_before))
avg_retrieval_after = float(np.mean(retrieval_after))
avg_clustering_before = float(np.nanmean(clustering_before))
avg_clustering_after = float(np.nanmean(clustering_after))

valid_clustering_batches = int(np.sum(~np.isnan(clustering_before)))

print(f"Batches evaluated: {len(retrieval_before)}")
print(f"Batches with valid clustering score: {valid_clustering_batches}")
print(
    f"Average gap norm   | before: {avg_gap_before:.4f} | after: {avg_gap_after:.4f}"
)
print(
    f"Average Retrieval@1| before: {avg_retrieval_before:.4f} | after: {avg_retrieval_after:.4f}"
)
print(
    f"Average V-measure  | before: {avg_clustering_before:.4f} | after: {avg_clustering_after:.4f}"
)


In [ ]:
clustering_original.item()

### Classification: baseline vs shifted

Train a `LinearProbingIndependentModalities` classifier (each modality independently) on:
1. **baseline** — original CLIP embeddings
2. **shifted** — embeddings after the embedding shift with λ=1

In [ ]:
# ── Helper utilities for classification ──────────────────────────────

def remap_labels(labels_gpu: torch.Tensor, lut_gpu: torch.Tensor) -> torch.Tensor:
    """Map original category ids to contiguous 0..C-1 class indices."""
    if labels_gpu.numel() == 0:
        return labels_gpu
    mapped = lut_gpu[labels_gpu]
    if (mapped < 0).any():
        bad = labels_gpu[mapped < 0][:20].cpu().numpy()
        raise ValueError(f"Labels missing from train label map: {bad}")
    return mapped


def make_optimizer_and_scheduler(model, train_loader, epochs=200, max_lr=3e-4, weight_decay=1e-4):
    optimizer = torch.optim.AdamW(model.parameters(), lr=max_lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=max_lr,
        epochs=epochs,
        steps_per_epoch=len(train_loader),
        pct_start=0.1,
        div_factor=25.0,
        final_div_factor=1e4,
        anneal_strategy="cos",
    )
    return optimizer, scheduler


def train_classifier_translation(
    scenario_name: str,
    scenario_mode: str,             # "baseline" or "shifted"
    train_loader,
    test_loader,
    gap_vector,
    lam: float,
    num_classes: int,
    feature_dim: int,
    device: str = "cuda",
    epochs: int = 200,
    patience: int = 20,
    max_lr: float = 3e-4,
    weight_decay: float = 1e-4,
    wandb_enabled: bool = True,
    wandb_project: str = "msrvtt_v2_translation_verification",
    wandb_group: str = "embedding_shift",
    extra_config: dict | None = None,
):
    """
    Train a LinearProbingIndependentModalities classifier for MSRVTT v2.
    Both modalities are stacked (2N samples) and classified independently.

    scenario_mode:
      - "baseline"  → train on original normalised embeddings
      - "shifted"   → train on shifted + re-normalised embeddings (λ applied)
    """
    scenario_dir = BASE_SAVE_DIR / scenario_name
    scenario_dir.mkdir(parents=True, exist_ok=True)

    model = LinearProbingIndependentModalities(d=feature_dim, num_classes=num_classes).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1).to(device)
    optimizer, scheduler = make_optimizer_and_scheduler(
        model, train_loader, epochs=epochs, max_lr=max_lr, weight_decay=weight_decay,
    )

    lut_gpu = LABEL_LUT.to(device)
    gap_t = torch.as_tensor(gap_vector, device=device, dtype=torch.float32) if scenario_mode == "shifted" else None

    run_config = {
        "scenario_name": scenario_name,
        "scenario_mode": scenario_mode,
        "feature_dim": int(feature_dim),
        "num_classes": int(num_classes),
        "lambda": float(lam),
        "epochs": int(epochs),
        "patience": int(patience),
        "max_lr": float(max_lr),
        "weight_decay": float(weight_decay),
    }
    if extra_config is not None:
        run_config.update(extra_config)

    run = None
    if wandb_enabled:
        run = wandb.init(
            project=wandb_project,
            group=wandb_group,
            name=scenario_name,
            config=run_config,
            reinit=True,
        )

    history = {
        "train": {"loss": [], "accuracy": [], "lr": [], **{g: [] for g in GAP_NAMES}},
        "test":  {"loss": [], "accuracy": [],            **{g: [] for g in GAP_NAMES}},
    }

    best_test_loss = float("inf")
    best_test_acc = 0.0
    best_loss_state = None
    best_acc_state = None
    epochs_no_improve = 0

    def _prepare(text_emb, vision_emb):
        """Normalise, optionally shift, stack both modalities."""
        X = F.normalize(text_emb, dim=-1)
        Y = F.normalize(vision_emb, dim=-1)
        if scenario_mode == "shifted" and gap_t is not None:
            eps = 1e-12
            X = X - lam * gap_t
            Y = Y - lam * gap_t
            X = X / (X.norm(dim=1, keepdim=True) + eps)
            Y = Y / (Y.norm(dim=1, keepdim=True) + eps)
        # stack both modalities: (2B, D)
        all_emb = torch.cat([X, Y], dim=0)
        return all_emb, X, Y

    def run_epoch(dataloader, phase):
        is_train = phase == "train"
        model.train() if is_train else model.eval()

        losses, accs, lrs = [], [], []
        gap_values = {g: [] for g in GAP_NAMES}

        for batch in tqdm(dataloader, desc=f"{scenario_name} | {phase}", leave=False):
            text_emb, vision_emb, labels = batch[:3]
            text_emb = text_emb.to(device, dtype=torch.float32, non_blocking=True)
            vision_emb = vision_emb.to(device, dtype=torch.float32, non_blocking=True)
            labels = torch.as_tensor(labels, device=device, dtype=torch.long)
            labels = remap_labels(labels, lut_gpu)

            if is_train:
                optimizer.zero_grad(set_to_none=True)

            all_emb, text_feat, vision_feat = _prepare(text_emb, vision_emb)
            all_labels = torch.cat([labels, labels], dim=0)  # duplicate labels for 2N

            logits = model(all_emb)
            loss = criterion(logits, all_labels)

            if is_train:
                loss.backward()
                clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()

            acc = (logits.argmax(dim=1) == all_labels).float().mean()
            losses.append(float(loss.item()))
            accs.append(float(acc.item()))
            if is_train:
                lrs.append(float(optimizer.param_groups[0]["lr"]))

            with torch.no_grad():
                for gap_name in GAP_NAMES:
                    gap_values[gap_name].append(
                        _to_scalar(compute_gap(gap_name, text_feat.detach(), vision_feat.detach(), iterations=None))
                    )

        mean_loss = float(np.mean(losses)) if losses else float("nan")
        mean_acc = float(np.mean(accs)) if accs else float("nan")
        mean_lr = float(np.mean(lrs)) if (is_train and lrs) else None
        mean_gaps = {g: float(np.mean(v)) for g, v in gap_values.items()}
        return mean_loss, mean_acc, mean_lr, mean_gaps

    # ── Training loop ────────────────────────────────────────────────
    for epoch in range(epochs):
        train_loss, train_acc, train_lr, train_gaps = run_epoch(train_loader, "train")
        with torch.no_grad():
            test_loss, test_acc, _, test_gaps = run_epoch(test_loader, "test")

        history["train"]["loss"].append(train_loss)
        history["train"]["accuracy"].append(train_acc)
        history["train"]["lr"].append(train_lr if train_lr is not None else float("nan"))
        for g in GAP_NAMES:
            history["train"][g].append(train_gaps[g])

        history["test"]["loss"].append(test_loss)
        history["test"]["accuracy"].append(test_acc)
        for g in GAP_NAMES:
            history["test"][g].append(test_gaps[g])

        if epoch % 10 == 0 or epoch == epochs - 1:
            print(
                f"[{scenario_name}] epoch {epoch+1}/{epochs} | "
                f"train_loss={train_loss:.4f}  train_acc={train_acc:.4f} | "
                f"test_loss={test_loss:.4f}  test_acc={test_acc:.4f}"
            )

        improved = False
        if test_loss < best_test_loss:
            best_test_loss = test_loss
            best_loss_state = copy.deepcopy(model.state_dict())
            improved = True
            torch.save({"epoch": epoch+1, "metric": "test_loss", "metric_value": best_test_loss,
                         "model_state_dict": best_loss_state, "config": run_config},
                        scenario_dir / "best_model_test_loss.pt")

        if test_acc > best_test_acc:
            best_test_acc = test_acc
            best_acc_state = copy.deepcopy(model.state_dict())
            improved = True
            torch.save({"epoch": epoch+1, "metric": "test_accuracy", "metric_value": best_test_acc,
                         "model_state_dict": best_acc_state, "config": run_config},
                        scenario_dir / "best_model_test_acc.pt")

        if improved:
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if run is not None:
            payload = {
                "epoch": epoch+1,
                "train/loss": train_loss, "train/accuracy": train_acc, "train/lr": train_lr,
                "test/loss": test_loss, "test/accuracy": test_acc,
                "best/test_loss": best_test_loss, "best/test_accuracy": best_test_acc,
                "early_stop/epochs_no_improve": epochs_no_improve,
            }
            for g in GAP_NAMES:
                payload[f"train/{g}"] = train_gaps[g]
                payload[f"test/{g}"] = test_gaps[g]
            wandb.log(payload)

        if epochs_no_improve >= patience:
            print(f"[{scenario_name}] early stop at epoch {epoch+1}/{epochs}")
            break

    # ── Final evaluation with best checkpoint ────────────────────────
    if best_acc_state is not None:
        model.load_state_dict(best_acc_state)
        best_ckpt = "test_accuracy"
    elif best_loss_state is not None:
        model.load_state_dict(best_loss_state)
        best_ckpt = "test_loss"
    else:
        best_ckpt = "last_epoch"

    with torch.no_grad():
        final_train_loss, final_train_acc, _, final_train_gaps = run_epoch(train_loader, "test")
        final_test_loss, final_test_acc, _, final_test_gaps = run_epoch(test_loader, "test")

    result = {
        "scenario_name": scenario_name,
        "scenario_mode": scenario_mode,
        "lambda": float(lam),
        "feature_dim": int(feature_dim),
        "best_checkpoint_metric": best_ckpt,
        "best_test_loss": float(best_test_loss),
        "best_test_acc": float(best_test_acc),
        "final_train_loss": float(final_train_loss),
        "final_train_acc": float(final_train_acc),
        "final_test_loss": float(final_test_loss),
        "final_test_acc": float(final_test_acc),
        "final_train_gaps": {k: float(v) for k, v in final_train_gaps.items()},
        "final_test_gaps": {k: float(v) for k, v in final_test_gaps.items()},
        "history": history,
        "checkpoint_dir": str(scenario_dir),
    }

    with open(scenario_dir / "summary.json", "w") as f:
        json.dump(result, f, indent=2)

    torch.save({"model_state_dict": model.state_dict(), "result": result, "config": run_config},
               scenario_dir / "best_model_loaded_final.pt")

    if run is not None:
        run.summary["best_test_loss"] = result["best_test_loss"]
        run.summary["best_test_acc"] = result["best_test_acc"]
        run.summary["final_train_acc"] = result["final_train_acc"]
        run.summary["final_test_acc"] = result["final_test_acc"]
        for g in GAP_NAMES:
            run.summary[f"final_train_{g}"] = result["final_train_gaps"][g]
            run.summary[f"final_test_{g}"] = result["final_test_gaps"][g]
        run.finish()

    print(f"[{scenario_name}] done | best_test_acc={result['best_test_acc']:.4f} | "
          f"final_test_acc={result['final_test_acc']:.4f}")
    return result

In [ ]:
# ── Baseline classification (original embeddings) ────────────────────

result_baseline = train_classifier_translation(
    scenario_name="msrvtt_v2__translation__baseline",
    scenario_mode="baseline",
    train_loader=train_loader,
    test_loader=test_loader,
    gap_vector=gap_vector,
    lam=0.0,
    num_classes=NUM_CLASSES,
    feature_dim=EMBEDDING_DIM,
    device=DEVICE,
    epochs=200,
    patience=20,
    max_lr=3e-4,
    weight_decay=1e-4,
    wandb_enabled=True,
    wandb_project="msrvtt_v2_translation_verification",
    wandb_group="embedding_shift",
    extra_config={"dataset": "msrvtt_v2", "classifier": "LinearProbingIndependentModalities"},
)

In [ ]:
# ── Shifted classification (λ = 1, full gap removal) ─────────────────

result_shifted = train_classifier_translation(
    scenario_name="msrvtt_v2__translation__shifted_lam1",
    scenario_mode="shifted",
    train_loader=train_loader,
    test_loader=test_loader,
    gap_vector=gap_vector,
    lam=1.0,
    num_classes=NUM_CLASSES,
    feature_dim=EMBEDDING_DIM,
    device=DEVICE,
    epochs=200,
    patience=20,
    max_lr=3e-4,
    weight_decay=1e-4,
    wandb_enabled=True,
    wandb_project="msrvtt_v2_translation_verification",
    wandb_group="embedding_shift",
    extra_config={"dataset": "msrvtt_v2", "classifier": "LinearProbingIndependentModalities"},
)

### Summary: baseline vs shifted (λ = 1)

In [ ]:
# ── Comparison table ──────────────────────────────────────────────────

rows = []
for res in [result_baseline, result_shifted]:
    row = {
        "scenario": res["scenario_name"],
        "mode": res["scenario_mode"],
        "λ": res["lambda"],
        "best_test_acc": res["best_test_acc"],
        "final_test_acc": res["final_test_acc"],
        "final_test_loss": res["final_test_loss"],
    }
    for g in GAP_NAMES:
        row[f"test_{g}"] = res["final_test_gaps"][g]
    rows.append(row)

df_summary = pd.DataFrame(rows)
display(df_summary.round(4))

# ── Training curves side by side ─────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for res, label in [(result_baseline, "baseline"), (result_shifted, "shifted λ=1")]:
    h = res["history"]
    axes[0].plot(h["train"]["loss"], label=f"{label} train")
    axes[0].plot(h["test"]["loss"], linestyle="--", label=f"{label} test")

    axes[1].plot(h["train"]["accuracy"], label=f"{label} train")
    axes[1].plot(h["test"]["accuracy"], linestyle="--", label=f"{label} test")

    axes[2].plot(h["test"]["RMG"], label=f"{label} RMG")

axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()
axes[2].set_title("Relative Modality Gap (test)"); axes[2].set_xlabel("Epoch"); axes[2].legend()

plt.tight_layout()
plt.savefig(BASE_SAVE_DIR / "training_curves_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nDone. Results saved to:", BASE_SAVE_DIR)

## Flickr30k — Embedding Shift (Translation)

This section reproduces the embedding shift experiment on Flickr30k.
We evaluate retrieval, gap metrics, and clustering before and after applying the translation shift with $\lambda = 1$.
Clustering is computed on a bounded subset to keep runtime manageable.

In [3]:
import pandas as pd
import torch
from collections import Counter
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

from analysis.modality_gap import compute_gap
from dataset.flickr30k.dataloader_embeddings_with_labels import EmbeddingsDatasetWithLabels
from metrics.clustering import _clustering_metrics_two_modalities_flickr30k

FLICKR_SPLIT_NAME = "flickr30k"
FLICKR_TEXT_INDEX = 0
FLICKR_MIN_CLASS_COUNT = 10
FLICKR_TEST_SIZE = 0.2
FLICKR_SPLIT_SEED = seed

flickr_dataset = EmbeddingsDatasetWithLabels(
    precomputed_dir=FLICKR_PRECOMPUTED_DIR,
    split_name=FLICKR_SPLIT_NAME,
    text_index=FLICKR_TEXT_INDEX,
)

flickr_class_counts = Counter()
for sample_idx in tqdm(range(len(flickr_dataset)), desc="Counting Flickr30k labels"):
    _, _, label_value = flickr_dataset[sample_idx]
    flickr_class_counts[int(label_value.item())] += 1

flickr_filtered_classes = {
    class_id for class_id, count in flickr_class_counts.items() if count >= FLICKR_MIN_CLASS_COUNT
}
flickr_filtered_indices = [
    sample_idx
    for sample_idx in range(len(flickr_dataset))
    if int(flickr_dataset[sample_idx][2].item()) in flickr_filtered_classes
]
flickr_filtered_dataset = Subset(flickr_dataset, flickr_filtered_indices)

flickr_filtered_labels = []
for sample_idx in flickr_filtered_indices:
    _, _, label_value = flickr_dataset[sample_idx]
    flickr_filtered_labels.append(int(label_value.item()))
flickr_filtered_labels = np.asarray(flickr_filtered_labels, dtype=np.int64)

flickr_train_idx, flickr_test_idx, _, _ = train_test_split(
    np.arange(len(flickr_filtered_dataset)),
    flickr_filtered_labels,
    test_size=FLICKR_TEST_SIZE,
    random_state=FLICKR_SPLIT_SEED,
    stratify=flickr_filtered_labels,
)

flickr_train_dataset = Subset(flickr_filtered_dataset, flickr_train_idx.tolist())
flickr_test_dataset = Subset(flickr_filtered_dataset, flickr_test_idx.tolist())

flickr_loader = DataLoader(
    flickr_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    generator=g,
)

print(f"Using model                     : {MODEL_NAME}")
print(f"Flickr precomputed dir          : {FLICKR_PRECOMPUTED_DIR}")
print(f"Flickr30k original samples      : {len(flickr_dataset)}")
print(f"Filtering threshold             : {FLICKR_MIN_CLASS_COUNT}")
print(f"Classes after filtering         : {len(flickr_filtered_classes)}")
print(f"Samples after filtering         : {len(flickr_filtered_dataset)}")
print(f"Train samples after split       : {len(flickr_train_dataset)}")
print(f"Test samples after split        : {len(flickr_test_dataset)}")
print(f"Embedding dim                   : {int(flickr_dataset[0][0].shape[-1])}")


[Loaded] 31783 samples from /mnt/media/emanuele/few_dimensions/dataset/flickr30k/precomputed_embeddings_with_labels/ViT-L-16-SigLIP2-256___webli


Counting Flickr30k labels: 100%|██████████| 31783/31783 [00:00<00:00, 147467.38it/s]


Using model                     : ViT-L-16-SigLIP2-256___webli
Flickr precomputed dir          : /mnt/media/emanuele/few_dimensions/dataset/flickr30k/precomputed_embeddings_with_labels/ViT-L-16-SigLIP2-256___webli
Flickr30k original samples      : 31783
Filtering threshold             : 10
Classes after filtering         : 465
Samples after filtering         : 30471
Train samples after split       : 24376
Test samples after split        : 6095
Embedding dim                   : 1024


In [4]:
def _compute_mean_shift(text_embeddings, vision_embeddings):
    return vision_embeddings.mean(dim=0) - text_embeddings.mean(dim=0)


def _gap_scalar(metric_name, x_text, x_vision):
    value = compute_gap(metric_name, x_text, x_vision, None)
    if isinstance(value, dict):
        if "text_vision" in value:
            value = value["text_vision"]
        else:
            value = next(iter(value.values()))
    if torch.is_tensor(value):
        value = value.item()
    return float(value)


def _normalize_rows(points, eps=1e-12):
    return points / (points.norm(p=2, dim=1, keepdim=True) + eps)


def _retrieval_at_k_chunked(text_embeddings, vision_embeddings, top_k=1, chunk_size=512):
    text_embeddings = torch.as_tensor(text_embeddings).float()
    vision_embeddings = torch.as_tensor(vision_embeddings).float()
    n_samples = text_embeddings.shape[0]
    correct = 0

    for start in tqdm(range(0, n_samples, chunk_size), desc=f"Retrieval@{top_k}", leave=False):
        stop = min(start + chunk_size, n_samples)
        similarities = text_embeddings[start:stop] @ vision_embeddings.T
        top_k_indices = torch.topk(similarities, k=top_k, dim=1).indices
        targets = torch.arange(start, stop).unsqueeze(1)
        correct += int((top_k_indices == targets).any(dim=1).sum().item())

    return correct / n_samples


flickr_text_batches, flickr_vision_batches, flickr_label_batches = [], [], []

for text_batch, vision_batch, label_batch in tqdm(flickr_loader, desc="Collecting filtered Flickr30k test embeddings"):
    flickr_text_batches.append(text_batch)
    flickr_vision_batches.append(vision_batch)
    flickr_label_batches.append(label_batch)

flickr_text = torch.cat(flickr_text_batches, dim=0)
flickr_vision = torch.cat(flickr_vision_batches, dim=0)
flickr_labels = torch.cat(flickr_label_batches, dim=0)

flickr_batch_rmg_original = [
    _gap_scalar("RMG", text_batch, vision_batch)
    for text_batch, vision_batch in zip(flickr_text_batches, flickr_vision_batches)
]
flickr_batch_rmg_original_mean = float(np.mean(flickr_batch_rmg_original))

flickr_gap = _compute_mean_shift(flickr_text, flickr_vision)
flickr_gap_norm_original = float(torch.norm(flickr_gap).item())

flickr_text_shifted = _normalize_rows(flickr_text + (flickr_gap / 2))
flickr_vision_shifted = _normalize_rows(flickr_vision - (flickr_gap / 2))
flickr_batch_rmg_shifted = []
for text_batch, vision_batch in zip(flickr_text_batches, flickr_vision_batches):
    batch_gap = _compute_mean_shift(text_batch, vision_batch)
    text_shifted_batch = _normalize_rows(text_batch + (batch_gap / 2))
    vision_shifted_batch = _normalize_rows(vision_batch - (batch_gap / 2))
    flickr_batch_rmg_shifted.append(_gap_scalar("RMG", text_shifted_batch, vision_shifted_batch))
flickr_batch_rmg_shifted_mean = float(np.mean(flickr_batch_rmg_shifted))
flickr_gap_shifted = _compute_mean_shift(flickr_text_shifted, flickr_vision_shifted)
flickr_gap_norm_shifted = float(torch.norm(flickr_gap_shifted).item())

flickr_cluster_count = int(torch.unique(flickr_labels).numel())

flickr_cluster_original = _clustering_metrics_two_modalities_flickr30k(
    flickr_text,
    flickr_vision,
    flickr_labels,
    n_clusters=flickr_cluster_count,
    random_state=SEED,
)
flickr_cluster_shifted = _clustering_metrics_two_modalities_flickr30k(
    flickr_text_shifted,
    flickr_vision_shifted,
    flickr_labels,
    n_clusters=flickr_cluster_count,
    random_state=SEED,
)

flickr_summary = pd.DataFrame(
    [
        {
            "state": "original",
            "gap_norm": flickr_gap_norm_original,
            "retrieval@1": _retrieval_at_k_chunked(flickr_text, flickr_vision, top_k=1),
            "retrieval@5": _retrieval_at_k_chunked(flickr_text, flickr_vision, top_k=5),
            "retrieval@10": _retrieval_at_k_chunked(flickr_text, flickr_vision, top_k=10),
            "RMG": flickr_batch_rmg_original_mean,
            "cosineTP": _gap_scalar("cosineTP", flickr_text, flickr_vision),
            "L2I": _gap_scalar("L2I", flickr_text, flickr_vision),
            "V-measure": float(flickr_cluster_original["V-measure"]),
        },
        {
            "state": "shifted_lam1",
            "gap_norm": flickr_gap_norm_shifted,
            "retrieval@1": _retrieval_at_k_chunked(flickr_text_shifted, flickr_vision_shifted, top_k=1),
            "retrieval@5": _retrieval_at_k_chunked(flickr_text_shifted, flickr_vision_shifted, top_k=5),
            "retrieval@10": _retrieval_at_k_chunked(flickr_text_shifted, flickr_vision_shifted, top_k=10),
            "RMG": flickr_batch_rmg_shifted_mean,
            "cosineTP": _gap_scalar("cosineTP", flickr_text_shifted, flickr_vision_shifted),
            "L2I": _gap_scalar("L2I", flickr_text_shifted, flickr_vision_shifted),
            "V-measure": float(flickr_cluster_shifted["V-measure"]),
        },
    ]
)

display(flickr_summary.round(4))
print(f"Filtered test samples used      : {len(flickr_test_dataset)}")
print(f"Clusters inferred from test set : {flickr_cluster_count}")

,state,gap_norm,retrieval@1,retrieval@5,retrieval@10,RMG,cosineTP,L2I,V-measure
0,original,18.2443,0.7659,0.9208,0.9534,0.8517,0.1791,25.6107,0.5036
1,shifted_lam1,0.2178,0.7121,0.8884,0.9316,0.7194,0.5263,0.9712,0.5331


Filtered test samples used      : 6095
Clusters inferred from test set : 465


## Export Results

Save compact embedding-shift results in `comparison/results`.


In [5]:
import json
import os
import numpy as np
import torch


def _json_safe_scalar(value):
    if value is None:
        return None
    if torch.is_tensor(value):
        if value.numel() == 1:
            value = value.item()
        else:
            raise ValueError("Expected scalar tensor during export.")
    if isinstance(value, np.generic):
        value = value.item()
    return value


def _compact_metric_dict(metrics):
    if metrics is None:
        return None
    return {str(k): float(_json_safe_scalar(v)) for k, v in metrics.items()}


def _compact_clustering(metrics):
    if metrics is None:
        return None
    keys = ["ARI", "NMI", "Homogeneity", "V-measure", "n_clusters", "n_classes"]
    compact = {}
    for key in keys:
        if key in metrics:
            value = _json_safe_scalar(metrics[key])
            compact[key] = int(value) if key.startswith("n_") else float(value)
    return compact


def _summary_frame_to_dict(df):
    if df is None:
        return None
    out = {}
    for _, row in df.iterrows():
        state = str(row["state"])
        out[state] = {
            str(k): float(_json_safe_scalar(v))
            for k, v in row.items()
            if k != "state"
        }
    return out


def _classification_summary(result):
    if result is None:
        return None
    return {
        "scenario_name": result.get("scenario_name"),
        "scenario_mode": result.get("scenario_mode"),
        "lambda": float(_json_safe_scalar(result.get("lambda"))),
        "feature_dim": int(_json_safe_scalar(result.get("feature_dim"))),
        "best_checkpoint_metric": result.get("best_checkpoint_metric"),
        "best_test_loss": float(_json_safe_scalar(result.get("best_test_loss"))),
        "best_test_acc": float(_json_safe_scalar(result.get("best_test_acc"))),
        "final_train_loss": float(_json_safe_scalar(result.get("final_train_loss"))),
        "final_train_acc": float(_json_safe_scalar(result.get("final_train_acc"))),
        "final_test_loss": float(_json_safe_scalar(result.get("final_test_loss"))),
        "final_test_acc": float(_json_safe_scalar(result.get("final_test_acc"))),
        "final_train_gaps": _compact_metric_dict(result.get("final_train_gaps")),
        "final_test_gaps": _compact_metric_dict(result.get("final_test_gaps")),
    }


def _msrvtt_embedding_shift_summary():
    required = [
        "gap_norm", "new_gap", "mean_batch_rmg_original", "mean_batch_rmg_new",
        "retrieval_1_original", "retrieval_1_new", "retrieval_5_new", "retrieval_10_new",
        "clustering_original", "clustering_new", "cosine_tp_new", "l2i_new",
    ]
    if not all(name in globals() for name in required):
        return None

    original_cluster = float(_json_safe_scalar(clustering_original)) if not isinstance(clustering_original, dict) else float(_json_safe_scalar(clustering_original.get("V-measure")))
    shifted_cluster = float(_json_safe_scalar(clustering_new)) if not isinstance(clustering_new, dict) else float(_json_safe_scalar(clustering_new.get("V-measure")))

    return {
        "original": {
            "gap_norm": float(_json_safe_scalar(gap_norm)),
            "retrieval@1": float(_json_safe_scalar(retrieval_1_original)),
            "RMG": float(_json_safe_scalar(mean_batch_rmg_original)),
            "V-measure": original_cluster,
        },
        "shifted_lam1": {
            "gap_norm": float(_json_safe_scalar(new_gap)),
            "retrieval@1": float(_json_safe_scalar(retrieval_1_new)),
            "retrieval@5": float(_json_safe_scalar(retrieval_5_new)),
            "retrieval@10": float(_json_safe_scalar(retrieval_10_new)),
            "RMG": float(_json_safe_scalar(mean_batch_rmg_new)),
            "cosineTP": float(_json_safe_scalar(cosine_tp_new)),
            "L2I": float(_json_safe_scalar(l2i_new)),
            "V-measure": shifted_cluster,
        },
    }


export_path = "/mnt/media/emanuele/few_dimensions/comparison/results/embedding_shift_evaluation_results.json"
os.makedirs(os.path.dirname(export_path), exist_ok=True)

if os.path.exists(export_path):
    with open(export_path, "r", encoding="utf-8") as f:
        export_payload = json.load(f)
else:
    export_payload = {}

model_payload = export_payload.get(MODEL_NAME, {})
model_payload["mscoco_imagenet"] = {
    "embedding_shift": _summary_frame_to_dict(globals().get("mscoco_summary")),
}
model_payload["msrvtt_v2"] = {
    "embedding_shift": _msrvtt_embedding_shift_summary(),
    "classification": {
        "baseline": _classification_summary(globals().get("result_baseline")),
        "shifted_lam1": _classification_summary(globals().get("result_shifted")),
    },
}
model_payload["flickr30k"] = {
    "embedding_shift": _summary_frame_to_dict(globals().get("flickr_summary")),
}
export_payload["seed"] = int(seed)
export_payload[MODEL_NAME] = model_payload

with open(export_path, "w", encoding="utf-8") as f:
    json.dump(export_payload, f, indent=2)

print(f"Saved embedding-shift results to {export_path} under model={MODEL_NAME}")


Saved embedding-shift results to /mnt/media/emanuele/few_dimensions/comparison/results/embedding_shift_evaluation_results.json under model=ViT-L-16-SigLIP2-256___webli
